In [1]:
data= 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [2]:
!wget = $data

--2026-09-26 17:02:37--  http://=/
Resolving = (=)... failed: Name or service not known.
wget: unable to resolve host address ‘=’
--2026-09-26 17:02:37--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.11’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.005s  

2026-09-26 17:02:38 (50.9 MB/s) - ‘course_lead_scoring_2026.csv.11’ saved [278746/278746]

FINISHED --2026-09-26 17:02:38--
Total wall clock time: 0.2s
Downloaded: 1 files, 272K in 0.005s (50.9 MB/s)


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
# Q1 What is the most frequent observation (mode) for the column industry?

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [8]:
categories=['lead_source','industry', 'employment_status', 'location', 'converted']
numerical=['annual_income','number_of_courses_viewed','interaction_count','lead_score']
df[categories] = df[categories].astype("category")

In [9]:
df.dtypes

lead_source                 category
industry                    category
employment_status           category
location                    category
annual_income                float64
number_of_courses_viewed       int64
interaction_count              int64
lead_score                   float64
converted                   category
dtype: object

In [10]:
# Add NaN as category
for col in categories:
    df[col] = df[col].cat.add_categories("NA").fillna("NA")

df[categories] = df[categories].fillna('NA')
df[numerical] = df[numerical].fillna(0)

# Check the categories with 
df['location'].cat.categories 

Index(['africa', 'asia', 'europe', 'north_america', 'south_america', 'NA'], dtype='str')

In [11]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [12]:
df['industry'].mode()

0    technology
Name: industry, dtype: category
Categories (7, str): ['education', 'finance', 'healthcare', 'manufacturing', 'retail', 'technology', 'NA']

In [13]:
# Q2: What are the two features that have the biggest correlation?

In [14]:
print(df[['interaction_count']].corrwith(df['lead_score']))
print(df[['number_of_courses_viewed']].corrwith(df['lead_score']))
print(df[['number_of_courses_viewed']].corrwith(df['interaction_count']))
print(df[['annual_income']].corrwith(df['interaction_count']))

interaction_count    0.915746
dtype: float64
number_of_courses_viewed    0.757204
dtype: float64
number_of_courses_viewed    0.721609
dtype: float64
annual_income    0.122842
dtype: float64


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X = df.copy()
del X['converted']
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Correlation matrix for the numerical features
#print(X[['interaction_count']].corrwith(X['lead_score']))
#print(X[['number_of_courses_viewed']].corrwith(X['lead_score']))
#print(X[['number_of_courses_viewed']].corrwith(X['interaction_count']))
#print(X[['annual_income']].corrwith(X['interaction_count']))
#X.head()

In [17]:
# Q3 Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
# Round the scores to 2 decimals using round(score, 2).

In [18]:
from sklearn.metrics import mutual_info_score

In [19]:
print(df['converted'].shape)
print(X.shape)

(5000,)
(5000, 8)


In [20]:
categories_without_converted=['lead_source','industry', 'employment_status', 'location']
for col in categories_without_converted:
    print(col, round(mutual_info_score(df_full_train['converted'], df_full_train[col]), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [21]:
# Q4:
# 1. Use one-hot encoding for the categorical variables
# 2. Fit the logistic regression on the training dataset with fit the model with these parameters:
#model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# 3. Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

In [22]:
# 1. Use one-hot encoding for the categorical variables

In [24]:
from sklearn.feature_extraction import DictVectorizer

In [68]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categories_without_converted + numerical].to_dict(orient='records')
val_dict = df_val[categories_without_converted + numerical].to_dict(orient='records')
y_train=df_train['converted'].values.astype(int)
y_val=df_val['converted'].values.astype(int)

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [28]:
# 2. Fit the logistic regression on the training dataset with fit the model with these parameters:
#model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
#Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

In [69]:
# The 'NA' category inside converted is giving problems, so I remove it
df_train['converted'] = df_train['converted'].cat.remove_unused_categories()
df_val['converted'] = df_val['converted'].cat.remove_unused_categories()

# Run model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)
accuracy_original=(y_val == churn_decision).mean().round(2)
accuracy_original

np.float64(0.64)

In [61]:
# See output
print(y_pred[:20])
print(churn_decision[:20])

print("Actual:")
print(y_val[:20])

print("Prediction counts:")
print(pd.Series(churn_decision).value_counts())

print("Actual counts:")
print(pd.Series(y_val).value_counts())

[0.61725691 0.44364108 0.60433996 0.64562246 0.52214159 0.77885722
 0.60698076 0.54481903 0.76770971 0.52765495 0.75038798 0.6470054
 0.56857815 0.51044645 0.59921425 0.46620745 0.62786484 0.69167143
 0.69082953 0.59101344]
[ True False  True  True  True  True  True  True  True  True  True  True
  True  True  True False  True  True  True  True]
Actual:
682     1
4312    1
647     0
878     1
777     0
2673    1
1423    1
1438    0
618     1
4914    1
4158    1
698     1
1395    1
4087    0
1282    1
4186    0
3905    1
4718    0
4286    1
2949    0
Name: converted, dtype: int64
Prediction counts:
True     874
False    126
Name: count, dtype: int64
Actual counts:
converted
1    597
0    403
Name: count, dtype: int64


In [ ]:
# Q5
#Let's find the least useful feature using the feature elimination technique.
#Train a model using the same features and parameters as in Q4 (without rounding).
#Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
#For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

In [ ]:
# feature elimination technique

In [65]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'annual_income': np.float64(-0.0),
 'employment_status=NA': np.float64(-0.001),
 'employment_status=employed': np.float64(0.007),
 'employment_status=self_employed': np.float64(-0.003),
 'employment_status=student': np.float64(-0.008),
 'employment_status=unemployed': np.float64(-0.006),
 'industry=NA': np.float64(-0.0),
 'industry=education': np.float64(-0.003),
 'industry=finance': np.float64(-0.001),
 'industry=healthcare': np.float64(-0.002),
 'industry=manufacturing': np.float64(-0.002),
 'industry=retail': np.float64(-0.002),
 'industry=technology': np.float64(-0.001),
 'interaction_count': np.float64(0.137),
 'lead_score': np.float64(0.011),
 'lead_source=NA': np.float64(0.0),
 'lead_source=events': np.float64(-0.009),
 'lead_source=organic_search': np.float64(0.007),
 'lead_source=paid_ads': np.float64(-0.014),
 'lead_source=referral': np.float64(0.009),
 'lead_source=social_media': np.float64(-0.004),
 'location=NA': np.float64(0.0),
 'location=africa': np.float64(-0.002),
 '

In [84]:
dv = DictVectorizer(sparse=False)

# Targets (doesn't change)
# The 'NA' category inside converted is giving problems, so I remove it
df_train['converted'] = df_train['converted'].cat.remove_unused_categories()
df_val['converted'] = df_val['converted'].cat.remove_unused_categories()

y_train = df_train['converted'].astype(int).values
y_val = df_val['converted'].astype(int).values

features = categories_without_converted + numerical
accuracies=[]
# Loop to get the accuracies of each model without 1 feature vs the original model with all the features
for i in range(len(features)):
    features2 = features[:i] + features[i+1:]
    train_dict = df_train[features2].to_dict(orient='records')
    val_dict = df_val[features2].to_dict(orient='records')

    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    # Run model
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    #accuracies.append((y_val == churn_decision).mean()
    print(features[i] + str((y_val == churn_decision).mean()))
    

lead_source0.642
industry0.645
employment_status0.645
location0.645
annual_income0.724
number_of_courses_viewed0.643
interaction_count0.601
lead_score0.644


In [79]:
print(accuracies)

[np.float64(0.64), np.float64(0.64), np.float64(0.64), np.float64(0.64), np.float64(0.72), np.float64(0.64), np.float64(0.6), np.float64(0.64)]


In [ ]:
# Q6 Train a regularized logistic regression.
#Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
#Train models using all the features as in Q4.
#Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

In [103]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categories_without_converted + numerical].to_dict(orient='records')
val_dict = df_val[categories_without_converted + numerical].to_dict(orient='records')
y_train=df_train['converted'].values.astype(int)
y_val=df_val['converted'].values.astype(int)

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
# The 'NA' category inside converted is giving problems, so I remove it
df_train['converted'] = df_train['converted'].cat.remove_unused_categories()
df_val['converted'] = df_val['converted'].cat.remove_unused_categories()

# Run model
C= [0.000001, 0.00001, 0.0001, 0.001]
from sklearn.linear_model import LogisticRegression
for c in C:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    #accuracy_original=(y_val == churn_decision).mean().round(2)
    print("C =",str(c), "has accuracy:" , str((y_val == churn_decision).mean()))

C = 1e-06 has accuracy: 0.598
C = 1e-05 has accuracy: 0.598
C = 0.0001 has accuracy: 0.613
C = 0.001 has accuracy: 0.645
